<a href="https://colab.research.google.com/github/arunabdevi/langchain_rag_hrassistant/blob/main/PROITBRIDGE_HR_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROITBRIDGE HR Assistant — RAG + Tool-Calling Agent (LangChain)

This notebook builds the assistant **step by step, in one linear flow**, so you can see the structure and the reasoning behind each decision before you split it into a modular project.

**What we're building:** a chatbot that answers HR questions from the PROITBRIDGE Employee Handbook using three routes:

1. **General policy questions** ("What is the casual leave policy?") → answered by **retrieval** (RAG) — the handbook text is searched and summarized.
2. **Calculations** ("How much gratuity after 7 years on Rs.60,000?") → answered by a **calculator tool** — real Python arithmetic, never the LLM's mental math.
3. **Specific validations** ("Can I take 4 days of CL in a row?") → answered by a **validator tool** — a hard-coded yes/no rule, never the LLM's guess.

A LangChain **agent** decides which route a question needs and calls the right tool. Every markdown cell below explains *why* the next code cell is written the way it is — including a few real bugs we hit and fixed along the way, because understanding those is as important as the working code.

**Before running:** upload `PROITBRIDGE_Employee_Handbook_2026_1.pdf` when Cell 3 asks for it, and have an OpenAI API key ready for Cell 2.

## Step 1 — Install dependencies

**What this cell does:** installs LangChain and the specific integration packages we need: `langchain-openai` (chat model + embeddings), `langchain-community` (the PDF loader), `langchain-chroma` (vector store), `pypdf` (the actual PDF parser LangChain calls), and `gradio` (for the chat UI at the end).

**Why these, specifically:** LangChain 1.x split most integrations into separate packages instead of one monolithic library, so each provider (OpenAI, Chroma, etc.) is its own pip install. Pinning `langchain>=1.4.0` matters because the API changed meaningfully at the 1.0 release — `create_agent` and `create_retriever_tool` moved to different import paths than in older tutorials you may find online (we hit this directly: `create_retriever_tool` used to live in `langchain.tools.retriever`, and in 1.x it's under `langchain_core.tools.retriever`).

In [1]:
!pip install -q langchain>=1.4.0 langchain-openai langchain-community langchain-chroma langchain-text-splitters
!pip install -q pypdf chromadb tiktoken gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.5 MB/s eta 0:00:00


## Step 2 — Set your OpenAI API key

**What this cell does:** prompts for your API key without echoing it to the notebook output, and sets it as an environment variable — this is how every LangChain OpenAI call (chat model + embeddings) picks up credentials automatically, without passing the key around explicitly in code.

**Why `getpass` instead of a plain variable:** typing `api_key = "sk-..."` directly into a cell means the key gets saved in the notebook's `.ipynb` file itself if you ever share or commit it. `getpass` keeps it out of the notebook's saved output.

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


## Step 3 — Upload the handbook PDF

**What this cell does:** opens Colab's file-upload widget and loads whatever PDF you provide into memory, then grabs its filename for the next step.

**Why upload rather than hard-coding a path:** Colab's runtime is a fresh, disposable container — there's no persistent filesystem with your PDF already on it. Uploading is the direct way to get a local file into that environment for this session.

In [3]:
from google.colab import files

uploaded = files.upload()  # select PROITBRIDGE_Employee_Handbook_2026_1.pdf
pdf_path = list(uploaded.keys())[0]
print(f"Loaded file: {pdf_path}")

Saving PROITBRIDGE_Employee_Handbook_2026_1.pdf to PROITBRIDGE_Employee_Handbook_2026_1.pdf
Loaded file: PROITBRIDGE_Employee_Handbook_2026_1.pdf


## Step 4 — Extract text from the PDF

**What this cell does:** uses LangChain's `PyPDFLoader` to read every page of the PDF and returns one `Document` object per page. We then join all pages into a single text blob.

**Why merge pages into one blob instead of keeping them separate:** if we chunk page-by-page later, every chunk boundary would be forced at a page break — even if that break falls in the middle of a sentence or a policy section. Merging first lets the *next* step (chunking) decide boundaries based on the document's actual structure (chapters, paragraphs), not on where the PDF happened to paginate.

**Why `PyPDFLoader` specifically:** this handbook is a WeasyPrint-exported PDF with a fairly clean internal structure — no OCR or heavy table-reconstruction is needed. `PyPDFLoader` handles it directly, including its tables, without extra tooling.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()
raw_text = "\n".join(page.page_content for page in pages)

print(f"Total pages: {len(pages)}")
print(f"Total characters: {len(raw_text)}")

/tmp/ipykernel_576/3550280312.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages: 57
Total characters: 104734


## Step 5 — Clean the extracted text

**What this cell does:** strips two kinds of PDF export noise before we chunk anything:
1. **Repeated header/footer boilerplate** — this handbook prints the same "PROITBRIDGE Employee Handbook / Version 6.0..." block and the same "Confidential ... Page N" footer on *every one of its 57 pages*.
2. **Bare bullet markers** — standalone `•` characters left behind with no text attached, from how the PDF exported its bulleted lists.

**Why cleaning has to happen *before* chunking, not after:** if this boilerplate stays in, it gets duplicated across dozens of chunks. That dilutes each chunk's embedding with identical, meaningless text, and can cause an *irrelevant* chunk to rank as a false match in retrieval purely because it shares boilerplate with the real answer. We confirmed this cleaning removes about **10% of the raw character count** with zero loss of actual policy content — verified directly by diffing before/after character counts on this exact handbook.

**Why regex instead of a PDF-library option to strip headers:** the header/footer text isn't marked as a "header" or "footer" in the PDF's structure — it's just repeated body text — so removing it means pattern-matching the literal repeated string, including handling a `\xa0` (non-breaking space) character that a plain-space regex would silently fail to match. (We hit this directly during development: a first attempt using ordinary spaces in the pattern didn't match at all, because the real text used `\xa0` around the `|` separators.)

In [5]:
import re

def clean_handbook_text(text: str) -> str:
    # Standalone bullet-marker lines ("• " with nothing else on the line)
    text = re.sub(r"^\s*\u2022\s*$", "", text, flags=re.MULTILINE)

    # Repeated title/version footer
    text = re.sub(
        r"PROITBRIDGE Employee Handbook\s*\n"
        r"Version 6\.0\s*\|\s*Effective 1 April 2026\s*\|\s*Internal & Confidential",
        "",
        text,
    )

    # Repeated confidentiality/page-number footer
    text = re.sub(
        r"PROITBRIDGE \| Employee Handbook \| Confidential\s*"
        r"PIB-HR-HB-2026-V6\s*Page\s*\d+",
        "",
        text,
    )

    # Collapse the blank-line runs left behind by the removals above
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

clean_text = clean_handbook_text(raw_text)

print(f"Before cleaning: {len(raw_text)} chars")
print(f"After cleaning:  {len(clean_text)} chars")
print(f"Reduction: {100 * (1 - len(clean_text) / len(raw_text)):.1f}%")

Before cleaning: 104734 chars
After cleaning:  94650 chars
Reduction: 9.6%


## Step 6 — Split the cleaned text into chunks

**What this cell does:** splits the cleaned handbook into retrieval-sized pieces using `RecursiveCharacterTextSplitter`, with a specific, ordered list of separators: try splitting on a **chapter break** first; if a piece is still too big, try a **paragraph break**; then a **line break**; then a sentence; then a word.

**Why this exact separator order, not just a fixed character count:** a naive fixed-size splitter cuts wherever the character count runs out, which can slice a table row in half or separate a rule from the heading that gives it context. Trying chapter breaks first means a chunk (almost) never straddles two unrelated policy chapters. Trying line breaks *before* sentence breaks matters specifically for this document because its tables render as one line per row — splitting on sentences first could still break a row apart, but a line-level split won't.

**Why `chunk_size=1200, chunk_overlap=200`:** verified directly against this handbook — the full Chapter 7 leave-entitlement table (a wide multi-row table) fits inside 1200 characters as one intact chunk, and 200 characters of overlap is enough that a fact stated right at a chunk boundary still appears in its neighbor's context too.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\nCHAPTER ", "\n\n", "\n", ". ", " "],
)

chunks = splitter.split_text(clean_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 112


## Step 7 — Sanity-check a chunk before trusting the pipeline

**What this cell does:** pulls out the specific chunk containing the Casual Leave section and prints it in full, checking two things: that it's free of the bullet/footer noise from Step 5, and that it wasn't cut off mid-sentence by Step 6.

**Why check this manually instead of assuming it worked:** during development, an early *truncated preview* of a retrieval result made it look like something was broken (a key sentence seemed to be missing), when the sentence was actually present just past where the preview cut off. The lesson: always print the **full** chunk content when verifying, not a truncated preview — a truncated view can manufacture a bug that isn't there, or hide one that is.

In [7]:
for c in chunks:
    if "Casual Leave (CL)" in c and "Maximum 3 consecutive days" in c:
        print(c)
        print(f"\n--- LENGTH: {len(c)} chars ---")
        print(f"Contains bullet noise: {'•' in c}")
        print(f"Contains footer noise: {'PIB-HR-HB-2026-V6' in c}")
        break

rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Minimum block of 0.5 day; there is no maximum, subject to approval.
Apply at least 7 working days in advance for 1–3 days, and 21 days in advance for 4 days or more.
Up to 45 days may be carried into the next year. Anything above 45 days lapses on 31 December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.

--- LENGTH: 912 chars ---
Contains bullet noise: False
Contains footer noise: False


## Step 8 — Embed the chunks and build a vector store

**What this cell does:** wraps each chunk as a LangChain `Document`, converts every chunk into a numeric vector with OpenAI's embedding model, and stores them all in a **Chroma** vector store — an in-process, no-server-needed database built for similarity search.

**Why Chroma specifically:** it runs embedded directly inside this notebook process with nothing extra to stand up (no separate database server), which is the right fit for a project of this size. It also supports persisting to disk, so a real project doesn't have to re-embed the whole handbook every time it starts.

**Why `text-embedding-3-small`:** a solid quality-to-cost tradeoff for a 57-page document — this doesn't need the largest embedding model to distinguish handbook sections from each other.

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

docs = [Document(page_content=chunk, metadata={"chunk_id": i}) for i, chunk in enumerate(chunks)]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="proitbridge_handbook",
)

print(f"Vector store built with {vectorstore._collection.count()} chunks")

Vector store built with 112 chunks


## Step 9 — Test raw retrieval before wrapping it as a tool

**What this cell does:** runs a real question straight against the vector store (no LLM, no agent yet) and prints the top 5 matching chunks, so we can visually confirm the right policy text actually comes back.

**Why `k=5` and not the more obvious `k=3`:** tested directly against this handbook — a query about Casual Leave sometimes ranked the *Earned Leave* section above the actual Casual Leave section, because EL is discussed more extensively elsewhere in the text and can look "topically closer" to an embedding model. Widening to `k=5` reliably keeps the correct chunk in the retrieved set, even when it isn't the single top-ranked result. This is worth testing here, before an agent is involved, because it isolates a retrieval-quality problem from an agent-reasoning problem — if retrieval itself is weak, no amount of prompt engineering later will fix it.

In [9]:
query = "Can casual leave be combined with earned leave?"
results = vectorstore.similarity_search(query, k=5)

for i, r in enumerate(results):
    print(f"--- Result {i+1} (chunk_id={r.metadata['chunk_id']}) ---")
    print(r.page_content[:300])
    print()

--- Result 1 (chunk_id=35) ---
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken 

--- Result 2 (chunk_id=34) ---
CHAPTER 07
Leave Policy
Every leave type, how much you get, how it accrues, and how to apply.
7.1 Leave Entitlement Summary
Leave Type Entitlement per
Year Accrual Carry Forward Encashable
Earned / Privilege Leave (EL) 18 days 1.5 days per completed
month
Up to 45 days Yes, on
separation
Casual Leav

--- Result 3 (chunk_id=42) ---
as leave. For CL and SL they are not.
Can leave be cancelled after
approval?
Yes, by either side. If the Company cancels approved leave and you incur a
non-refundable cost, the Company reimburses it.
Can I take leave during notice
period?
Only with written approval, and it does not extend or offset 

-

## Step 10 — Wrap the vector store as a LangChain tool

**What this cell does:** wraps the vector store's retriever in `create_retriever_tool`, which turns it into a named tool (`search_handbook`) an agent can call just like any other function — this is **Route 1** from the intro: general policy lookups.

**Why the import is `langchain_core.tools.retriever` and not `langchain.tools.retriever`:** this is a real gotcha in LangChain 1.x. Many older tutorials (and LangChain's own pre-1.0 docs) show `create_retriever_tool` importing from `langchain.tools.retriever`, which no longer exists after the 1.0 restructuring — it now lives under `langchain_core`. If you hit `ImportError: cannot import name 'create_retriever_tool'`, this is almost always why.

**Why the tool's description matters as much as its code:** the agent decides *which* tool to call based on reading each tool's name and description, not its implementation. A vague description ("searches documents") would make the agent's routing unreliable — the description below explicitly tells it when to use this tool versus a calculator/validator.

In [10]:
from langchain_core.tools.retriever import create_retriever_tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

retriever_tool = create_retriever_tool(
    retriever,
    name="search_handbook",
    description=(
        "Search the PROITBRIDGE Employee Handbook for policy information — "
        "leave rules, compensation, benefits, conduct, security, working hours, "
        "performance management, or any other HR policy. Use this whenever a "
        "question needs a fact, rule, number, or entitlement from the handbook "
        "in general terms, with no specific numbers or scenario to compute or "
        "validate."
    ),
)

# quick sanity check
print(retriever_tool.invoke({"query": "What is the notice period for a confirmed employee?"})[:400])

CHAPTER 20
Employee Separation
Resignation, notice periods, handover, exit formalities and full-and-final settlement.
20.1 Types of Separation
Type Initiated By Notice Settlement
Resignation Employee 60 days (confirmed) / 30
days (probation)
Full and final within 45 days of the
last working day
Termination without cause Company 60 days or salary in lieu Full and final including notice pay
Terminat


## Step 11 — The HR calculator and validator tools

This is **Route 2** (calculations) and **Route 3** (validations) from the intro. Each tool below is a plain Python function decorated with `@tool`, which makes it callable by the agent. All arithmetic and all yes/no policy decisions live here as tested code — **never** left to the LLM to compute or judge in free text.

**One design rule applies to every tool below, and it came from a real, confirmed bug — worth understanding before reading the code:**

> **A required input the user didn't actually supply must be typed `Optional[...] = None`, and the function must explicitly detect and report that as a "missing input" result — never silently defaulted to a placeholder like `0`, `False`, or a made-up date.**

Here's why this rule exists. Early testing asked the agent: *"I'm submitting a reimbursement claim 20 days after returning, is it valid?"* — with no amount mentioned. The agent **invented** `amount=0`, called the validator, got back "invalid: amount must be > 0", and confidently reported that as the answer — a completely fabricated basis for a real answer.

The first fix attempted was a **system-prompt instruction** ("never invent a value for a missing parameter"). **This did not work** — the exact same fabrication happened again immediately after. Tool-calling LLMs are trained to fill in every declared parameter of a function's schema, and a paragraph of prompt instructions competing against that trained behavior lost.

The fix that actually worked was in the **code itself**: make the parameter `Optional[float] = None`, and have the function check for `None` first and return an explicit "I'm missing this" result. This works because Pydantic (which backs every `@tool` function's schema) then makes `None` a legitimate, well-typed value the model can pass — and the tool's own docstring tells it that doing so is correct, not a fallback.

**Every tool below follows this pattern.** Each also range-checks its inputs (no negative years of service, no rating outside 1–5, valid date formats) — a second, related bug we found was a tool accepting a nonsensical-but-well-typed input (like `months_in_current_grade=-5`) and it happening to produce the right answer *by coincidence of its sign* rather than because it was actually validated.

### Tools 1–2 — Leave request validator, probation end calculator

**`validate_leave_request`** answers *"can I take X days of [leave type]"* — Route 3. It hard-codes the actual multi-condition rules (EL cannot be taken during probation; EL needs 7 or 21 days notice depending on length; CL caps at 3 consecutive days) so the agent never has to reason through these conditions itself and risk missing one.

**`calculate_probation_end`** answers *"when does my probation end"* — Route 2. Note `joining_date: Optional[str] = None` — this is the fabrication-safety pattern from above, applied to its first real tool.

In [11]:
from datetime import datetime, timedelta
from typing import Optional
from langchain_core.tools import tool


@tool
def validate_leave_request(
    leave_type: str,
    days: int,
    is_on_probation: bool,
    advance_days: int,
) -> dict:
    """Check whether a SPECIFIC leave request is allowed or prohibited.

    Use this whenever the employee asks whether a particular leave request is
    allowed, permitted, valid, or prohibited -- e.g. "Can I take 4 days of
    casual leave in a row?". For a general policy question like "What is the
    casual leave policy?" use search_handbook instead.

    leave_type: "EL" (Earned), "CL" (Casual) or "SL" (Sick).
    days: number of leave days requested.
    is_on_probation: True if the employee is still on probation.
    advance_days: working days of advance notice given.
    """
    leave_type = leave_type.upper().strip()

    if days <= 0:
        return {"valid": False, "reason": "Leave days must be greater than 0.",
                "policy_reference": "Chapter 7"}

    if leave_type not in {"EL", "CL", "SL"}:
        return {"valid": False, "reason": "Unsupported leave type. Use EL, CL or SL.",
                "policy_reference": "Chapter 7.1"}

    if leave_type == "EL":
        if is_on_probation:
            return {"valid": False,
                    "reason": "Earned Leave cannot be availed during probation. "
                              "It accrues but can only be taken after confirmation.",
                    "policy_reference": "Chapter 4.5 / Chapter 7.2"}
        if days <= 3 and advance_days < 7:
            return {"valid": False,
                    "reason": "Earned Leave of 1-3 days needs at least 7 working days notice.",
                    "policy_reference": "Chapter 7.2"}
        if days >= 4 and advance_days < 21:
            return {"valid": False,
                    "reason": "Earned Leave of 4 or more days needs at least 21 working days notice.",
                    "policy_reference": "Chapter 7.2"}

    if leave_type == "CL" and days > 3:
        return {"valid": False, "reason": "Casual Leave cannot exceed 3 consecutive days.",
                "policy_reference": "Chapter 7.3"}

    return {"valid": True, "reason": f"{leave_type} request for {days} day(s) is valid.",
            "policy_reference": "Chapter 7"}


@tool
def calculate_probation_end(joining_date: Optional[str] = None) -> dict:
    """Calculate the probation end date for a specific joining date (YYYY-MM-DD).

    Use for questions like "when does my probation end if I joined 2026-01-01".
    Handbook rule: standard probation is 90 calendar days from joining, unless
    the appointment letter specifies otherwise (e.g., 180 days for Manager+).

    If the employee has not stated their joining date, pass joining_date=None
    rather than guessing one -- this tool will ask for it explicitly.
    """
    if joining_date is None:
        return {"valid": None, "reason": "Joining date was not provided.",
                "next_step": "Please ask the employee for their joining date (YYYY-MM-DD).",
                "policy_reference": "Chapter 4.5"}

    try:
        start = datetime.strptime(joining_date.strip(), "%Y-%m-%d")
    except ValueError:
        return {"valid": False, "reason": "joining_date must be in YYYY-MM-DD format.",
                "policy_reference": "Chapter 4.5"}

    end = start + timedelta(days=90)
    return {"valid": True, "joining_date": start.strftime("%Y-%m-%d"),
            "probation_days": 90, "probation_end_date": end.strftime("%Y-%m-%d"),
            "note": "Standard probation is 90 calendar days unless your appointment "
                    "letter specifies otherwise (e.g., 180 days for Manager and above).",
            "policy_reference": "Chapter 4.5"}

print("Tools 1-2 defined.")

Tools 1-2 defined.


### Tools 3–4 — Payroll deadline, reimbursement claim validator

**`check_payroll_deadline`** answers *"when is my salary credited"* — Route 2. Same `Optional[str] = None` pattern for `date`.

**`validate_reimbursement_claim`** is the tool from the fabrication bug story above — `amount: Optional[float] = None` is the actual fix. It also demonstrates something equally important: **honest uncertainty**. The handbook says claims should be submitted within 15 days, and that claims older than 45 days need special approval — but it never says what happens in the **16–45 day gap in between**. Rather than guessing which way that ambiguity resolves, the tool returns `valid: None` with an explicit explanation and defers to HR/Finance. A tool that silently picked a side here would be worse than one that admits the handbook doesn't say.

In [12]:
@tool
def check_payroll_deadline(date: Optional[str] = None) -> dict:
    """Report payroll cut-off and salary/payslip dates for a specific date (YYYY-MM-DD).

    Use for questions like "when is salary credited" or "what is the payroll deadline".
    Handbook pay cycle: attendance 21st-20th, cut-off the 20th, salary credited
    the last working day of the month before 6 PM IST, payslip by the 1st of
    the next month.

    If the employee has not stated which month/date to check, pass date=None
    rather than guessing a date -- this tool will ask for it explicitly.
    """
    if date is None:
        return {"valid": None, "reason": "No date was provided.",
                "next_step": "Please ask the employee which month or date they want the payroll deadline for.",
                "policy_reference": "Chapter 8.1"}

    try:
        d = datetime.strptime(date.strip(), "%Y-%m-%d")
    except ValueError:
        return {"valid": False, "reason": "date must be in YYYY-MM-DD format.",
                "policy_reference": "Chapter 8.1"}

    cutoff_day = 20
    inputs_open = d.day <= cutoff_day
    first_next_month = datetime(d.year + 1, 1, 1) if d.month == 12 else datetime(d.year, d.month + 1, 1)
    last_day = first_next_month - timedelta(days=1)

    return {"valid": True, "date": d.strftime("%Y-%m-%d"), "attendance_period": "21st to 20th",
            "payroll_cutoff": d.replace(day=cutoff_day).strftime("%Y-%m-%d"),
            "payroll_inputs_status": ("Open — inputs accepted until the 20th." if inputs_open
                                       else "Closed — the 20th cut-off has passed; inputs move to the next cycle."),
            "salary_credit_date": last_day.strftime("%Y-%m-%d") + " (last working day before 6 PM IST)",
            "payslip_by": first_next_month.strftime("%Y-%m-%d"),
            "policy_reference": "Chapter 8.1"}


@tool
def validate_reimbursement_claim(
    days_since_return: int,
    has_approved_travel_request: bool,
    has_required_documents: bool,
    amount: Optional[float] = None,
) -> dict:
    """Check whether a SPECIFIC travel reimbursement claim is valid.

    Use for questions like "can I claim reimbursement 20 days after returning".
    Handbook rules: submit within 15 days; claims older than 45 days need
    Finance Head approval; bills above Rs.500 need legible scans; an approved
    travel request is required; claims above Rs.50,000 settle within 10 working days.

    If the employee has not stated the claim amount, pass amount=None rather
    than guessing a value -- this tool will ask for it explicitly.
    """
    if amount is None:
        return {"valid": None, "reason": "Claim amount was not provided.",
                "next_step": "Please ask the employee for the exact claim amount before this can be validated.",
                "policy_reference": "Chapter 17.4"}

    if amount <= 0:
        return {"valid": False, "reason": "Claim amount must be greater than 0.",
                "next_step": "Enter the actual claim amount.", "policy_reference": "Chapter 17.4"}

    if not has_approved_travel_request:
        return {"valid": False, "reason": "An approved travel request is required before claiming.",
                "next_step": "Raise a travel request in the HRMS and get manager approval first.",
                "policy_reference": "Chapter 17.1 / 17.4"}

    if amount > 500 and not has_required_documents:
        return {"valid": False, "reason": "Bills above Rs.500 require legible scans.",
                "next_step": "Attach legible scans of every bill above Rs.500.", "policy_reference": "Chapter 17.4"}

    if days_since_return > 45:
        return {"valid": False, "reason": "Claim is older than 45 days.",
                "next_step": "This needs Finance Head approval and may be declined.", "policy_reference": "Chapter 17.4"}

    # The handbook is silent on the 16-45 day window -- we say so, we don't guess.
    if days_since_return > 15:
        return {"valid": None,
                "reason": "The handbook does not explicitly define the approval status for a claim "
                          "submitted 16-45 days after returning.",
                "next_step": "Please confirm with HR/Finance.", "policy_reference": "Chapter 17.4"}

    settlement = ("Finance settles within 10 working days (claim above Rs.50,000)."
                  if amount > 50000 else "Finance settles with the next payroll cycle.")
    return {"valid": True, "reason": "Claim is within the 15-day window and has the required approvals.",
            "next_step": f"Manager approves within 3 working days. {settlement}", "policy_reference": "Chapter 17.4"}

print("Tools 3-4 defined.")

Tools 3-4 defined.


### Tools 5–6 — Remote work allowance, travel daily allowance

Both are Route 2 lookups against a fixed rate table (no dates, no ambiguity — just a value keyed by category). `calculate_remote_work_allowance` normalizes its input (`.lower().strip()`, collapsing `_`/spaces to `-`) so `"REMOTE FIRST"`, `"remote_first"`, and `"remote-first"` all resolve to the same key — tested directly and confirmed this handles real phrasing variation correctly.

`calculate_travel_daily_allowance` range-checks `grade` (must be 1–5) rather than trusting it — this is the same "malformed-but-present input" safeguard mentioned in Step 11's intro.

In [13]:
@tool
def calculate_remote_work_allowance(work_model: str) -> dict:
    """Return the internet allowance for a SPECIFIC work model.

    Use for questions like "calculate my remote-first allowance".
    Handbook rules: hybrid = Rs.1,500/month, remote-first = Rs.2,500/month.
    """
    model = work_model.lower().strip().replace("_", "-").replace(" ", "-")
    allowances = {"hybrid": 1500, "remote-first": 2500}

    if model not in allowances:
        return {"valid": False, "reason": "Unsupported work model. Supported: hybrid, remote-first.",
                "policy_reference": "Chapter 18.3"}

    return {"valid": True, "work_model": model, "internet_allowance_per_month": allowances[model],
            "currency": "INR", "policy_reference": "Chapter 18.3"}


@tool
def calculate_travel_daily_allowance(location_type: str, grade: int) -> dict:
    """Return the travel daily allowance for a SPECIFIC location and grade.

    Use for questions like "travel allowance for Grade 3 in a metro".
    location_type: metro, other_india, asia_middle_east or europe_americas.
    grade: employee grade from 1 to 5.
    """
    location = location_type.lower().strip()
    rates = {
        "metro": {"1-2": (1200, "INR"), "3": (1600, "INR"), "4-5": (2200, "INR")},
        "other_india": {"1-2": (900, "INR"), "3": (1200, "INR"), "4-5": (1600, "INR")},
        "asia_middle_east": {"1-2": (45, "USD"), "3": (60, "USD"), "4-5": (80, "USD")},
        "europe_americas": {"1-2": (65, "USD"), "3": (85, "USD"), "4-5": (110, "USD")},
    }

    if location not in rates:
        return {"valid": False, "reason": "Unsupported location.", "policy_reference": "Chapter 17.3"}
    if grade not in (1, 2, 3, 4, 5):
        return {"valid": False, "reason": "Grade must be between 1 and 5.", "policy_reference": "Chapter 17.3"}

    band = "1-2" if grade in (1, 2) else "3" if grade == 3 else "4-5"
    amount, currency = rates[location][band]
    return {"valid": True, "location_type": location, "grade": grade, "daily_allowance": amount,
            "currency": currency, "policy_reference": "Chapter 17.3"}

print("Tools 5-6 defined.")

Tools 5-6 defined.


### Tool 7 — Earned Leave balance projector

This tool has the most interesting history of the whole set, worth understanding rather than skimming.

An **earlier version of this project had two separate tools**: one that projected EL balance given a joining date and a target date, and a second, simpler one that just multiplied a raw month-count by the accrual rate. That overlap caused a real bug: when a user said *"I joined 2026-01-01, how much EL will I have accrued?"* (a joining date, but no target date), the agent picked the simpler tool — because it *could* answer with just a month-count — and **fabricated** a plausible-looking month-count (`6`) to make the call work, rather than recognizing it actually needed the other tool (which would have correctly asked for the missing target date).

Adding an `Optional[int] = None` to the simpler tool would only have patched the symptom. **The actual fix was removing the redundant tool entirely** and merging both use cases into the one tool below — including a special, deliberate value: passing the literal string `"today"` for `target_date` means *"as of right now,"* which is a well-defined choice the agent can make deliberately, not a guess.

**Lesson generalized:** two tools that can both plausibly answer the same question is itself a source of fabrication risk, independent of whether either tool individually handles missing inputs correctly.

In [14]:
@tool
def project_earned_leave_balance(
    is_confirmed: bool,
    joining_date: Optional[str] = None,
    target_date: Optional[str] = None,
    days_already_taken: int = 0,
) -> dict:
    """Project the Earned Leave (EL) balance on a specific target date.

    Use this for ALL EL accrual/projection questions -- "how much EL will I
    accrue after 6 months", "how much EL have I accrued so far", "how much EL
    will I have by 30 June". There is no separate month-count-only tool.

    joining_date: YYYY-MM-DD. If not stated, pass None -- do not guess it.
    target_date: YYYY-MM-DD, OR the literal string "today" if asking about the
      balance so far / right now. If neither a date nor "today" is implied,
      pass None -- this tool will ask. Do not guess a date or a month count.
    is_confirmed: True if the employee has cleared probation (EL accrues during
      probation but cannot be availed until confirmation).
    days_already_taken: EL days already used, netted off the projected balance.
    Handbook rule: EL accrues at 1.5 days per completed month of service.
    """
    missing = []
    if joining_date is None:
        missing.append("joining_date")
    if target_date is None:
        missing.append("target_date (or say 'today' for the balance so far)")
    if missing:
        return {"valid": None, "reason": f"Missing required input(s): {', '.join(missing)}.",
                "next_step": "Please ask the employee for the missing information.",
                "policy_reference": "Chapter 7.2"}

    try:
        start = datetime.strptime(joining_date.strip(), "%Y-%m-%d")
    except ValueError:
        return {"valid": False, "reason": "joining_date must be in YYYY-MM-DD format.",
                "policy_reference": "Chapter 7.2"}

    resolved_note = None
    if target_date.strip().lower() == "today":
        target = datetime.now()
        resolved_note = f"Resolved 'today' to {target.strftime('%Y-%m-%d')}."
    else:
        try:
            target = datetime.strptime(target_date.strip(), "%Y-%m-%d")
        except ValueError:
            return {"valid": False, "reason": "target_date must be YYYY-MM-DD, or the literal word 'today'.",
                    "policy_reference": "Chapter 7.2"}

    if target < start:
        return {"valid": False, "reason": "target_date cannot be before joining_date.",
                "policy_reference": "Chapter 7.2"}

    completed_months = (target.year - start.year) * 12 + (target.month - start.month)
    if target.day < start.day:
        completed_months -= 1
    completed_months = max(completed_months, 0)

    accrued = completed_months * 1.5
    net_balance = accrued - days_already_taken

    availability = ("Availed EL is netted off above; the balance is available to use." if is_confirmed
                     else "This EL has accrued but CANNOT be availed until the employee is confirmed (Chapter 4.5).")
    note = availability + (f" {resolved_note}" if resolved_note else "") + \
        " This projection ignores the 45-day annual carry-forward cap."

    return {"valid": True, "target_date": target.strftime("%Y-%m-%d"), "completed_months": completed_months,
            "accrual_rate_per_month": 1.5, "accrued_el_days": accrued, "days_already_taken": days_already_taken,
            "projected_el_balance": net_balance, "note": note, "policy_reference": "Chapter 7.2"}

print("Tool 7 defined.")

Tool 7 defined.


### Tools 8–10 — Gratuity, notice shortfall, promotion eligibility

**`calculate_gratuity`** and **`calculate_notice_shortfall`** both surface an **explicit `"assumption"` field** in their output. Neither tool's formula is fully specified by the handbook: gratuity's "15 days' basic per year" doesn't itself state the standard 26-working-day divisor used to get a *daily* rate, and notice-shortfall's "salary in lieu" doesn't specify how to convert a monthly salary into a daily rate either. Rather than silently applying an assumption and presenting the result as pure handbook fact, both tools name the assumption they used — and the system prompt (next step) instructs the agent to relay that assumption to the user, not hide it.

**`check_promotion_eligibility`** is where the "malformed-but-present input" bug from Step 11's intro was actually found: a test sent `months_in_current_grade=-5`, and the *original* version of this tool (without the guard clause below) produced the *correct-looking* answer ("not eligible") purely by coincidence — a negative number is still less than 18. It also silently accepted `latest_rating=47` as "≥ 3, so it passes." Both guard clauses below were added specifically because of that finding.

In [15]:
@tool
def calculate_gratuity(
    years_of_service: int,
    monthly_basic_salary: float,
    death_or_permanent_disablement: bool = False,
) -> dict:
    """Calculate gratuity payable for a SPECIFIC length of service and basic salary.

    Use for questions like "how much gratuity after 7 years on Rs.60,000 basic".
    Handbook rule: payable under the Payment of Gratuity Act, 1972 after 5
    years of continuous service, at 15 days' basic salary per completed year.
    The 5-year condition is waived on death or permanent disablement.
    """
    if years_of_service < 0 or monthly_basic_salary <= 0:
        return {"valid": False, "reason": "years_of_service must be >= 0 and monthly_basic_salary must be > 0.",
                "policy_reference": "Chapter 9.2"}

    if years_of_service < 5 and not death_or_permanent_disablement:
        return {"valid": False,
                "reason": f"Not eligible. Requires 5 years of continuous service (only {years_of_service} completed).",
                "policy_reference": "Chapter 9.2 / 20.4"}

    # Standard formula: (monthly basic / 26) x 15 x years. The handbook states
    # "15 days' basic per year" but not the 26-day divisor -- flagged explicitly.
    gratuity_amount = (monthly_basic_salary / 26) * 15 * years_of_service

    return {"valid": True, "years_of_service": years_of_service, "monthly_basic_salary": monthly_basic_salary,
            "gratuity_amount": round(gratuity_amount, 2),
            "assumption": "Uses the standard 26-working-day divisor per the Payment of Gratuity Act, 1972 "
                          "-- the handbook itself does not restate this divisor.",
            "note": "Paid within 30 days of separation.", "policy_reference": "Chapter 9.2 / 20.4"}


@tool
def calculate_notice_shortfall(is_confirmed: bool, days_notice_served: int, monthly_salary: float) -> dict:
    """Calculate the notice-period shortfall and salary-in-lieu for a resignation.

    Use for "I served only 20 days notice, how much salary in lieu do I owe".
    Handbook rule: 60 days notice for confirmed employees, 30 days on probation.
    """
    if days_notice_served < 0 or monthly_salary <= 0:
        return {"valid": False, "reason": "days_notice_served must be >= 0 and monthly_salary must be > 0.",
                "policy_reference": "Chapter 4.5 / 20.1"}

    required_days = 60 if is_confirmed else 30
    shortfall_days = max(required_days - days_notice_served, 0)
    salary_in_lieu = round((monthly_salary / 30) * shortfall_days, 2)

    return {"valid": True, "required_notice_days": required_days, "days_notice_served": days_notice_served,
            "shortfall_days": shortfall_days, "salary_in_lieu": salary_in_lieu,
            "assumption": "Uses a 30-day month to derive the daily rate -- the handbook does not itself "
                          "specify this, or whether 'salary' means basic or gross. Confirm with HR/Finance.",
            "policy_reference": "Chapter 4.5 / 20.1"}


@tool
def check_promotion_eligibility(months_in_current_grade: int, latest_rating: int,
                                 has_active_disciplinary_proceeding: bool) -> dict:
    """Check whether an employee meets the MINIMUM eligibility bar for promotion.

    Handbook rule: minimum eligibility is 18 months in the current grade, a
    rating of 3+ in the latest cycle, and no active disciplinary proceeding.
    Meeting this bar does not guarantee promotion -- it only allows a case to
    be submitted.
    """
    if months_in_current_grade < 0:
        return {"valid": False, "reason": "months_in_current_grade cannot be negative.",
                "policy_reference": "Chapter 10.4"}
    if not (1 <= latest_rating <= 5):
        return {"valid": False, "reason": "latest_rating must be between 1 and 5.",
                "policy_reference": "Chapter 10.3 / 10.4"}

    unmet = []
    if months_in_current_grade < 18:
        unmet.append(f"Only {months_in_current_grade} months in current grade (18 required).")
    if latest_rating < 3:
        unmet.append(f"Latest rating is {latest_rating} (3 or above required).")
    if has_active_disciplinary_proceeding:
        unmet.append("An active disciplinary proceeding is disqualifying.")

    return {"valid": True, "eligible": len(unmet) == 0, "unmet_conditions": unmet,
            "note": "Meeting minimum eligibility only allows a manager to submit a promotion case.",
            "policy_reference": "Chapter 10.4"}

print("Tools 8-10 defined.")

Tools 8-10 defined.


### Tool registry

**What this cell does:** collects all 10 tools into one list (`HR_TOOLS`) that gets bound to the agent, plus a name→function dictionary (`TOOL_REGISTRY`) useful for testing a tool directly without going through the agent.

**Why a registry dict in addition to the list:** it lets you call `TOOL_REGISTRY["calculate_gratuity"].invoke({...})` directly to unit-test a single tool's logic in isolation — exactly how the bugs described above were actually found, by testing tools individually before ever involving the LLM.

In [16]:
HR_TOOLS = [
    validate_leave_request,
    calculate_probation_end,
    check_payroll_deadline,
    validate_reimbursement_claim,
    calculate_remote_work_allowance,
    calculate_travel_daily_allowance,
    project_earned_leave_balance,
    calculate_gratuity,
    calculate_notice_shortfall,
    check_promotion_eligibility,
]

TOOL_REGISTRY = {t.name: t for t in HR_TOOLS}

print(f"{len(HR_TOOLS)} HR tools registered:")
for t in HR_TOOLS:
    print(" -", t.name)

10 HR tools registered:
 - validate_leave_request
 - calculate_probation_end
 - check_payroll_deadline
 - validate_reimbursement_claim
 - calculate_remote_work_allowance
 - calculate_travel_daily_allowance
 - project_earned_leave_balance
 - calculate_gratuity
 - calculate_notice_shortfall
 - check_promotion_eligibility


## Step 12 — The system prompt (routing logic)

**What this cell does:** defines the instructions that tell the agent how to choose between Route 1 (search_handbook), Route 2 (calculators), and Route 3 (validators) — and encodes the safety rules that came directly from the bugs above.

**Why the prompt gives worked examples of near-identical phrasings routed differently:** *"How many consecutive CL days are allowed?"* (Route 1 — asks the general rule) and *"Can I take 4 days of casual leave in a row?"* (Route 3 — asks about one specific request) are semantically close but need completely different handling. Without explicit contrastive examples, this is exactly the kind of distinction a model can blur.

**Why "missing input" and "uncertain/undefined" get their own dedicated sections, not just one line each:** these map directly to the two most consequential bugs found during development — parameter fabrication (Step 11's intro) and the reimbursement tool's honest `valid: None` for its undefined 16–45 day window. The prompt explicitly tells the agent to relay that uncertainty to the user rather than resolve it with its own guess, and to state any tool's stated `"assumption"` (like gratuity's 26-day divisor) rather than presenting it as unqualified fact.

**Important caveat repeated from Step 11:** this prompt is necessary for *routing* (deciding which tool to call), but it was **not sufficient on its own** to stop fabrication — that had to be enforced in the tools' code (the `Optional[...] = None` pattern). Treat the prompt as the routing layer and the tool code as the safety layer; don't rely on the prompt alone for correctness.

In [17]:
HR_SYSTEM_PROMPT = (
    "You are the PROITBRIDGE HR Assistant. You answer PROITBRIDGE employees using the\n"
    "official PROITBRIDGE Employee Handbook and a set of deterministic HR tools.\n\n"
    "Before you answer, choose exactly ONE route.\n\n"

    "ROUTE 1 - GENERAL POLICY / KNOWLEDGE -> call search_handbook.\n"
    "  The question asks what a policy or rule IS, in general terms, with no specific\n"
    "  numbers, dates, or scenario to plug in.\n"
    "  e.g. 'What is the casual leave policy?'\n"
    "       'How many consecutive CL days are allowed?'\n"
    "       'What is the dress code on Fridays?'\n\n"

    "ROUTE 2 - SPECIFIC CALCULATION -> call the matching calculation tool.\n"
    "  'When does my probation end if I joined 2026-01-01?'      -> calculate_probation_end\n"
    "  'When is salary credited this month?'                    -> check_payroll_deadline\n"
    "  'Calculate my remote-first allowance.'                    -> calculate_remote_work_allowance\n"
    "  'Travel allowance for Grade 3 in a metro?'                -> calculate_travel_daily_allowance\n"
    "  'How much EL have I accrued so far / by a future date?'   -> project_earned_leave_balance\n"
    "  'How much gratuity after 7 years on a Rs.60,000 basic?'   -> calculate_gratuity\n"
    "  'I served only 20 days notice, how much salary in lieu?'  -> calculate_notice_shortfall\n"
    "  Use project_earned_leave_balance for ALL EL accrual questions -- there is no\n"
    "  separate month-count-only tool. 'So far'/'now' with no future date -> target_date='today'.\n\n"

    "ROUTE 3 - SPECIFIC VALIDATION / ELIGIBILITY / PERMISSION -> call the matching validation tool.\n"
    "  'Can I take 4 days of casual leave in a row?'          -> validate_leave_request\n"
    "  'Can I claim reimbursement 20 days after returning?'   -> validate_reimbursement_claim\n"
    "  'Am I eligible for promotion given [specific facts]?'  -> check_promotion_eligibility\n\n"

    "KEY DISTINCTIONS:\n"
    "  'How many consecutive CL days are allowed?' asks the RULE                     -> ROUTE 1.\n"
    "  'Can I take 4 days of casual leave in a row?' asks about a SPECIFIC request   -> ROUTE 3.\n"
    "  'How is gratuity calculated?' asks the RULE                                   -> ROUTE 1.\n"
    "  'How much gratuity after 7 years on Rs.60,000?' asks to COMPUTE               -> ROUTE 2.\n\n"

    "If the employee asks whether a specific leave request, reimbursement claim, or\n"
    "promotion case is allowed, valid, or eligible, you MUST call the matching\n"
    "validation tool. Never answer from memory or handbook text alone.\n\n"

    "If the employee gives specific numbers, dates, or amounts and asks for a result,\n"
    "you MUST call the matching calculation tool. Never perform the arithmetic yourself.\n\n"

    "MISSING REQUIRED INFORMATION:\n"
    "  Before calling a calculation or validation tool, check whether the employee's\n"
    "  message actually supplies every input the tool needs. If one or more required\n"
    "  inputs are missing, do NOT invent, guess, assume, or default a value for them\n"
    "  (e.g. do not assume amount=0, is_on_probation=False). Call the tool with the\n"
    "  missing input omitted (it will return a clear 'missing input' result), or ask\n"
    "  the employee directly.\n\n"

    "MULTI-STEP QUESTIONS:\n"
    "  Some questions require multiple tool calls in sequence, using one tool's output\n"
    "  as input to the next, before giving one combined final answer.\n\n"

    "HANDLING UNCERTAIN OR UNDEFINED CASES:\n"
    "  Some tools may return valid: null/None -- the handbook does not define the rule\n"
    "  for that scenario. Relay this honestly; do not resolve the ambiguity yourself.\n"
    "  If a tool result includes an 'assumption' field, state that assumption to the\n"
    "  employee rather than presenting it as an unqualified handbook fact.\n\n"

    "GROUNDING RULES:\n"
    "- Use the handbook context returned by search_handbook for policy answers.\n"
    "- Do not invent company policies, numbers, or thresholds not present in the\n"
    "  handbook or returned by a tool in THIS conversation turn.\n"
    "- Always cite the policy_reference field from the tool's output (e.g. 'per Chapter 7.2').\n"
    "- Keep the response concise and professional; answer in plain language, not raw JSON.\n"
    "- Do not call the same tool twice with the same arguments."
)

print(f"System prompt: {len(HR_SYSTEM_PROMPT)} characters")

System prompt: 3966 characters


## Step 13 — Assemble the agent

**What this cell does:** creates the chat model, combines the 10 HR tools with the retriever tool from Step 10 into one list, and binds everything (model + tools + system prompt) together with `create_agent`.

**Why `create_agent` and not `AgentExecutor`/`create_tool_calling_agent`:** those were the pre-1.0 LangChain pattern for building a tool-calling loop by hand. LangChain 1.x's `create_agent` is a thin wrapper over LangGraph that handles the same loop (LLM decides → tool runs → result goes back to the LLM → repeat until a final answer) with one function call. If you find older tutorials using `AgentExecutor`, that's the version this project intentionally moved away from.

**Why `temperature=0`:** this assistant's value comes from *consistency* — the same question should route to the same tool and get the same answer every time. Creative variation (which higher temperature encourages) works against that goal here.

In [18]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

all_tools = HR_TOOLS + [retriever_tool]

agent = create_agent(model=llm, tools=all_tools, system_prompt=HR_SYSTEM_PROMPT)

print(f"Agent assembled with {len(all_tools)} tools.")

Agent assembled with 11 tools.


## Step 14 — Test all three routes end-to-end

**What this cell does:** sends one question per route (plus a couple of edge cases) through the full agent, and prints which tool it called with what arguments, alongside the final answer — this is the same kind of test that surfaced every bug discussed above, so it's worth running in full rather than just trusting the code looks right.

**Why print the tool call, not just the final answer:** the final answer alone can *look* correct even when the underlying tool call was wrong (this is exactly how the `amount=0` fabrication bug was first caught — the answer sounded plausible, but the tool call revealed a fabricated value). Always inspect what the agent actually called, especially while you're still building trust in a new agent.

In [19]:
from langchain_core.messages import HumanMessage

test_queries = [
    "What is the dress code on Fridays?",                                                          # Route 1
    "How much gratuity will I get after 7 years on a Rs.60,000 basic?",                              # Route 2
    "Can I take 4 days of casual leave in a row?",                                                  # Route 3
    "How much EL have I accrued so far? I joined on 2026-01-01 and I'm confirmed.",                  # merged EL tool, "today"
    "I'm submitting a reimbursement claim 20 days after returning, I have an approved travel "
    "request and all documents. Is my claim valid?",                                                 # missing amount -- must NOT fabricate
]

for q in test_queries:
    print("=" * 80)
    print("Q:", q)
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print("  TOOL CALL:", tc["name"], tc["args"])
    print("A:", result["messages"][-1].content)
    print()

Q: What is the dress code on Fridays?
  TOOL CALL: search_handbook {'query': 'dress code on Fridays'}
A: On Fridays, the dress code at PROITBRIDGE is casual across the company, except where you have a client commitment. This means you can wear smart casual attire, which includes clean, neat, and presentable clothing such as jeans, t-shirts, kurtas, and sneakers. 

For more formal occasions, such as client meetings, the dress code would shift to business casual, which requires collared shirts or formal tops, trousers or formal skirts, and closed shoes. 

This information is outlined in Chapter 12.3 of the Employee Handbook.

Q: How much gratuity will I get after 7 years on a Rs.60,000 basic?
  TOOL CALL: calculate_gratuity {'years_of_service': 7, 'monthly_basic_salary': 60000}
A: After 7 years of service with a basic salary of Rs. 60,000, you will receive approximately Rs. 242,307.69 as gratuity. This calculation is based on the standard 26-working-day divisor per the Payment of Gratuit

## Step 15 — Chat UI (Gradio)

**What this cell does:** wraps the agent in a small function and hands it to `gr.ChatInterface`, which renders a full chat UI **inline in this notebook** when you run the cell.

**Why Gradio instead of Streamlit here:** Streamlit needs its own separate server process and a tunnel (like `ngrok` or `localtunnel`) to be viewable from a Colab notebook, which adds real friction just to see a chat window. Gradio's `launch()` renders directly inside Colab's output cell with zero extra setup, which is the better fit for *this* environment. (The GitHub-ready modular version of this project uses Streamlit instead, which is more standard for a project you'll run locally or deploy — but for exploring the logic in Colab, Gradio is the pragmatic choice.)

**Why wrap `agent.invoke` in a plain function instead of calling it directly in the UI definition:** `gr.ChatInterface` expects a function with the signature `(message, history) -> response` — this thin wrapper adapts our agent's `invoke` call to that exact interface without changing anything about the agent itself.

**Note:** `share=True` gives you a temporary public URL if you want to test this on your phone or share it with someone else during this session — omit it if you only need the inline view.

In [20]:
import gradio as gr

def chat_fn(message, history):
    result = agent.invoke({"messages": [HumanMessage(content=message)]})
    return result["messages"][-1].content

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🧭 PROITBRIDGE HR Assistant",
    description=(
        "Ask about leave, payroll, benefits, or any other PROITBRIDGE HR policy. "
        "Answers come from the official Employee Handbook and a set of policy "
        "calculators — missing details are asked for, never guessed."
    ),
    examples=[
        "What is the notice period for a confirmed employee?",
        "Can I take 4 days of casual leave in a row?",
        "How much gratuity will I get after 7 years on a Rs.60,000 basic?",
        "How much EL will I have by 30 June if I joined 1 January and I'm confirmed?",
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0cfd70f2d563a1e6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Where to go from here

This notebook is deliberately linear so the *logic* is visible top to bottom. Once you're comfortable with how each piece works, the natural next step is splitting this into a modular project:

- `config.py` — settings (paths, model names, chunk size)
- `ingest.py` — Steps 4–8 (load, clean, chunk, embed)
- `tools.py` — Step 11 (all 10 tools)
- `rag.py` — Step 10 (the retriever tool)
- `prompts.py` — Step 12 (the system prompt)
- `chain.py` — Step 13 (agent assembly)
- `assistant.py` — a thin class exposing `.ask(question)`, hiding the agent/message-list details
- Two frontends built on top of that one class: a Streamlit web app, and/or this notebook's Gradio UI

That modular version — including a full pytest suite encoding every bug described above as a permanent regression test, and a business-logic documentation file — is the GitHub-ready project delivered earlier in this conversation.